In [ ]:
# Import necessary libraries
from huggingface_hub import login
from datasets import load_dataset
import pandas as pd
import numpy as np
import torch
import transformers
import torch.nn as nn

# Load the dataset
dataset = load_dataset("CodeHima/TOS_DatasetV3")

dataset


DatasetDict({
    train: Dataset({
        features: ['sentence', 'unfairness_level'],
        num_rows: 7945
    })
    validation: Dataset({
        features: ['sentence', 'unfairness_level'],
        num_rows: 1050
    })
    test: Dataset({
        features: ['sentence', 'unfairness_level'],
        num_rows: 1045
    })
})

In [23]:
# Convert to a pandas DataFrame for easier visualization and manipulation
df = pd.DataFrame(dataset['train'])

# Check shape — should show (rows, columns)
print(df.shape)

# Check the first few rows to see what we're working with
print(df.head())

# Check label distribution for data imbalance
print(df['sentence'].value_counts())
print(df['unfairness_level'].value_counts())


(7945, 2)
                                            sentence    unfairness_level
0  these terms and any rights and licenses grante...        clearly_fair
1  the user is responsible for all damages liabil...      clearly_unfair
2  no refunds for downtime  the company is not li...  potentially_unfair
3  ea recommends that parents and guardians famil...        clearly_fair
4  the company can limit or restrict your ability...  potentially_unfair
sentence
                                                                                                                                                                                                                    59
limitation of liability                                                                                                                                                                                              4
indemnity                                                                                                        

In [24]:
# Check for nulls
print(df.isnull().sum())

# Check for rows where the sentence is irregularly short so we can see what we need to clean up
print(df[df['sentence'].str.len() < 20])

sentence            0
unfairness_level    0
dtype: int64
              sentence unfairness_level
11            feedback     clearly_fair
13     user guidelines   clearly_unfair
45            services     clearly_fair
47                         clearly_fair
51    terms of service   clearly_unfair
...                ...              ...
7863          accurate     clearly_fair
7868                ix   clearly_unfair
7884           licence     clearly_fair
7920        assignment     clearly_fair
7929   billing support   clearly_unfair

[475 rows x 2 columns]


In [25]:
def clean_df(df):
    df = df[df['sentence'].str.len() >= 20].copy()
    df['sentence'] = df['sentence'].str.strip()
    df = df[df['sentence'].str.len() > 0]
    return df

In [26]:
# Properly split the dataset into train, validation, and test sets
df_train = pd.DataFrame(dataset['train'])
df_val = pd.DataFrame(dataset['validation'])
df_test = pd.DataFrame(dataset['test'])

# Clean up the dataframes for each split
df_train = clean_df(df_train)
df_val = clean_df(df_val)
df_test = clean_df(df_test)

# Map text labels to integers for model training
label_map = {
    'clearly_fair': 0,
    'potentially_unfair': 1,
    'clearly_unfair': 2
}

# Apply label mapping to all three splits
df_train['label'] = df_train['unfairness_level'].map(label_map)
df_val['label'] = df_val['unfairness_level'].map(label_map)
df_test['label'] = df_test['unfairness_level'].map(label_map)

# Sanity check the label distribution in training set
print(df_train['label'].value_counts())

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

label
0    3694
1    1952
2    1824
Name: count, dtype: int64
Train: 7470 | Val: 971 | Test: 960


In [27]:
# Run this cell to save the pandas dfs to csv so they can be used in the other notebooks

df_train.to_csv('df_train.csv', index=False)
df_val.to_csv('df_val.csv', index=False)
df_test.to_csv('df_test.csv', index=False)